In [ ]:
import torch

from ml_ds.dataset import ERA5ZarrDataset
from ml_ds.network import LightningModule
from ml_ds.train_CNN import (
    ACTIVATION,
    CRITERION,
    DATA_ROOT,
    DEFAULT_INPUT_FILE,
    DEFAULT_RESULTS_DIR,
    DEFAULT_STATS_FILE,
    DROPOUT_RATE,
    LEARNING_RATE,
    N_BLOCKS,
    N_FILTERS,
    NORMALIZATION,
    NUM_WORKERS,
    _resolve_data_path,
    initialize_model,
)

In [ ]:
checkpoint_path = DEFAULT_RESULTS_DIR / "checkpoints" / "last.ckpt"
print(f"Checkpoint path: {checkpoint_path}")
data_root = DATA_ROOT.expanduser()
input_file = _resolve_data_path(DEFAULT_INPUT_FILE.expanduser(), data_root)
print(f"Input file: {input_file}")
stats_file = _resolve_data_path(DEFAULT_STATS_FILE.expanduser(), data_root)
print(f"Stats file: {stats_file}")

In [ ]:
batch_size = int(getattr(dataset, "time_chunk_size", 0))
if batch_size <= 0:
    raise ValueError("Could not infer a valid time chunk size from the input dataset.")
print(f"Using batch size = {batch_size} based on the dataset's time chunk size.")

In [ ]:
model = initialize_model(
    in_channels=len(dataset.input_vars),
    out_channels=len(dataset.target_vars),
    n_filters=N_FILTERS,
    n_blocks=N_BLOCKS,
    normalization=NORMALIZATION,
    activation=ACTIVATION,
    dropout_rate=DROPOUT_RATE,
)

# ---- Load Lightning checkpoint ----
network = LightningModule.load_from_checkpoint(
    checkpoint_path=checkpoint_path,
    model=model,
    train_dataset=dataset,
    val_dataset=None,
    test_dataset=None,
    lr=LEARNING_RATE,
    batch_size=batch_size,
    num_workers=NUM_WORKERS,
    criterion=CRITERION,
    enable_validation=False,
    map_location="cpu",
)
network.eval()

In [ ]:
record_index = 105
dataset = ERA5ZarrDataset(input_file, stats_file)
if len(dataset) == 0:
    raise ValueError("Input dataset is empty.")
if record_index < 0 or record_index >= len(dataset):
    raise IndexError(f"record_index={record_index} is out of range [0, {len(dataset) - 1}]")

In [ ]:
x_norm, y_norm = dataset[record_index]
with torch.no_grad():
    pred_norm = network(x_norm.unsqueeze(0)).squeeze(0).cpu()

In [ ]:
# De-normalize prediction and target
pred = pred_norm * dataset.target_stds.cpu() + dataset.target_means.cpu()
target = y_norm.cpu() * dataset.target_stds.cpu() + dataset.target_means.cpu()
x = x_norm.cpu() * dataset.input_stds.cpu() + dataset.input_means.cpu()

In [ ]:
print(f"Target vars: {dataset.target_vars}")
print(f"Prediction shape: {tuple(pred.shape)}")
print(f"Input vars: {dataset.input_vars}")
print(f"Input shape: {tuple(x.shape)}")

In [ ]:
import matplotlib.pyplot as plt

x_plot = x[5, :, :].numpy()
target_plot = target[1, :, :].numpy()
pred_plot = pred[1, :, :].numpy()

vmin = min(x_plot.min(), target_plot.min(), pred_plot.min())
vmax = max(x_plot.max(), target_plot.max(), pred_plot.max())

fig, axes = plt.subplots(1, 3, figsize=(12, 4))

im0 = axes[0].imshow(x_plot, vmin=vmin, vmax=vmax)
axes[0].set_title("x[5, :, :]")
axes[0].axis("off")

im1 = axes[1].imshow(target_plot, vmin=vmin, vmax=vmax)
axes[1].set_title("target[1, :, :]")
axes[1].axis("off")

im2 = axes[2].imshow(pred_plot, vmin=vmin, vmax=vmax)
axes[2].set_title("pred[1, :, :]")
axes[2].axis("off")

fig.colorbar(im2, ax=axes, fraction=0.03, pad=0.02)
plt.subplots_adjust(wspace=0.05, left=0.01, right=0.92, top=0.88, bottom=0.02)
plt.show()